# Bài 4 — Tic-Tac-Toe: Heuristic Alpha-Beta với depth cutoff

Protocol: seed 42; T1–T4 dùng cùng trạng thái Bài 3; node counter được reset mỗi search; thời gian là thời gian search, không gồm hiển thị.

In [1]:
import math
import time
from functools import lru_cache

import numpy as np
import pandas as pd

from tictactoe import actions, empty_board, other, result, utility

SEED = 42
np.random.seed(SEED)

def make_board(marks):
    board = empty_board()
    for index, symbol in marks.items():
        board[index] = symbol
    return board

STATES = {
    "T1 — X sắp thắng": make_board({0: "x", 1: "o", 3: "o", 4: "x"}),
    "T2 — X cần hòa": make_board({0: "x", 1: "o", 2: "x", 4: "o"}),
    "T3 — bất lợi": make_board({0: "o", 2: "x", 8: "o"}),
    "T4 — bàn rỗng": empty_board(),
}

def board_text(board):
    return "/".join("".join(board[i:i+3]).replace(" ", "·") for i in range(0, 9, 3))


In [2]:
def ordered_actions(board, move_ordering=False):
    legal = actions(board)
    if not move_ordering:
        return legal
    # Centre, corners, then edges; deterministic tie-breaker makes timing comparable.
    priority = [4, 0, 2, 6, 8, 1, 3, 5, 7]
    return [move for move in priority if move in legal]

def exact_search(board, player="x", alpha_beta=True, move_ordering=False):
    """Minimax/alpha-beta with a fresh node counter and perf_counter timing."""
    nodes = 0
    started = time.perf_counter()

    def visit(state, to_move, alpha=-math.inf, beta=math.inf):
        nonlocal nodes
        nodes += 1
        terminal_value = utility(state, player)
        if terminal_value is not None:
            return terminal_value
        maximizing = to_move == player
        value = -math.inf if maximizing else math.inf
        for move in ordered_actions(state, move_ordering):
            child_value = visit(result(state, to_move, move), other(to_move), alpha, beta)
            if maximizing:
                value = max(value, child_value)
                alpha = max(alpha, value)
            else:
                value = min(value, child_value)
                beta = min(beta, value)
            if alpha_beta and alpha >= beta:
                break
        return value

    root_value = visit(board, player)
    elapsed_s = time.perf_counter() - started
    root_scores = []
    # Reuse exact child values only for reporting the complete set of optimal moves.
    # This is excluded from elapsed_s/node counts of the algorithm above.
    for move in ordered_actions(board, move_ordering):
        root_scores.append((move, exact_value(result(board, player, move), other(player), player)))
    best_moves = tuple(move for move, value in root_scores if value == root_value)
    return {"move": best_moves[0] if best_moves else None, "optimal_moves": best_moves,
            "value": root_value, "nodes": nodes, "elapsed_s": elapsed_s}

@lru_cache(maxsize=None)
def _cached_value(state_tuple, to_move, player):
    state = list(state_tuple)
    terminal_value = utility(state, player)
    if terminal_value is not None:
        return terminal_value
    values = [_cached_value(tuple(result(state, to_move, move)), other(to_move), player)
              for move in actions(state)]
    return max(values) if to_move == player else min(values)

def exact_value(state, to_move, player):
    return _cached_value(tuple(state), to_move, player)


In [3]:
def lines_of_three(state):
    grid = np.asarray(state).reshape(3, 3)
    return list(grid) + list(grid.T) + [grid.diagonal(), np.fliplr(grid).diagonal()]

def evaluate_ttt(state, player="x"):
    terminal_value = utility(state, player)
    if terminal_value is not None:
        return float(terminal_value)
    opponent = other(player)
    score = 0.0
    for line in lines_of_three(state):
        own, opp = int(np.sum(line == player)), int(np.sum(line == opponent))
        if opp == 0:
            score += (0.06, 0.20)[own - 1] if own else 0.0
        elif own == 0:
            score -= (0.06, 0.20)[opp - 1] if opp else 0.0
    if state[4] == player: score += 0.05
    if state[4] == opponent: score -= 0.05
    return float(np.clip(score, -0.99, 0.99))

def heuristic_search(board, player="x", cutoff=None, move_ordering=True):
    nodes = 0
    started = time.perf_counter()
    def visit(state, to_move, depth, alpha=-math.inf, beta=math.inf):
        nonlocal nodes
        nodes += 1
        terminal_value = utility(state, player)
        if terminal_value is not None: return float(terminal_value)
        if cutoff is not None and depth >= cutoff: return evaluate_ttt(state, player)
        maximizing = to_move == player
        value, best = (-math.inf, None) if maximizing else (math.inf, None)
        for move in ordered_actions(state, move_ordering):
            child = visit(result(state, to_move, move), other(to_move), depth + 1, alpha, beta)
            if (maximizing and child > value) or (not maximizing and child < value):
                value, best = child, move
            if maximizing: alpha = max(alpha, value)
            else: beta = min(beta, value)
            if alpha >= beta: break
        return value if depth else (value, best)
    value, move = visit(board, player, 0)
    return {"move": move, "value": float(value), "nodes": nodes,
            "elapsed_s": time.perf_counter() - started}

def play_vs_random(agent, agent_symbol, games=100, seed=SEED):
    rng = np.random.default_rng(seed)
    outcomes = {"win": 0, "draw": 0, "loss": 0}
    total_agent_s, agent_moves = 0.0, 0
    for _ in range(games):
        board, to_move = empty_board(), "x"
        while utility(board, agent_symbol) is None:
            if to_move == agent_symbol:
                found = agent(board, to_move)
                move = found["move"]
                total_agent_s += found["elapsed_s"]; agent_moves += 1
            else:
                move = int(rng.choice(actions(board)))
            board = result(board, to_move, move)
            to_move = other(to_move)
        value = utility(board, agent_symbol)
        outcomes[{1: "win", 0: "draw", -1: "loss"}[value]] += 1
    return {**outcomes, "games": games, "agent_moves": agent_moves,
            "total_agent_s": total_agent_s,
            "mean_s_per_agent_move": total_agent_s / agent_moves}


## Heuristic

Mỗi hàng/cột/chéo còn mở được thưởng/phạt theo số quân; ô giữa có trọng số nhỏ. Trạng thái terminal luôn là -1, 0, 1; non-terminal bị chặn trong [-0.99, 0.99].

In [4]:
examples = pd.DataFrame([{"Trạng thái": name, "Bàn cờ": board_text(board), "h(X)": round(evaluate_ttt(board), 3)}
                         for name, board in STATES.items()])
examples

,Trạng thái,Bàn cờ,h(X)
0,T1 — X sắp thắng,xo·/ox·/···,0.31
1,T2 — X cần hòa,xox/·o·/···,-0.19
2,T3 — bất lợi,o·x/···/··o,-0.26
3,T4 — bàn rỗng,···/···/···,0.00


## So sánh full Alpha-Beta, cutoff = 2, cutoff = 4

In [5]:
rows = []
for name, board in STATES.items():
    full = heuristic_search(board, cutoff=None)
    full_optimal = exact_search(board, alpha_beta=True, move_ordering=True)["optimal_moves"]
    for label, cutoff in (("full", None), ("cutoff=2", 2), ("cutoff=4", 4)):
        found = full if cutoff is None else heuristic_search(board, cutoff=cutoff)
        rows.append({"Trạng thái": name, "Cấu hình": label, "Nước đi": found["move"],
                     "Giá trị": round(found["value"], 3), "Số nút": found["nodes"],
                     "Thời gian (ms)": round(found["elapsed_s"] * 1_000, 3),
                     "Khớp nước tối ưu full": found["move"] in full_optimal})
cutoff_results = pd.DataFrame(rows)
cutoff_results

,Trạng thái,Cấu hình,Nước đi,Giá trị,Số nút,Thời gian (ms),Khớp nước tối ưu full
0,T1 — X sắp thắng,full,2,1.00,44,1.090,True
1,T1 — X sắp thắng,cutoff=2,8,1.00,14,1.037,True
2,T1 — X sắp thắng,cutoff=4,2,1.00,36,1.843,True
3,T2 — X cần hòa,full,7,0.00,149,3.293,True
4,T2 — X cần hòa,cutoff=2,7,-0.13,26,2.007,True
5,T2 — X cần hòa,cutoff=4,7,-0.05,120,7.140,True
6,T3 — bất lợi,full,4,-1.00,51,2.617,True
7,T3 — bất lợi,cutoff=2,4,-0.23,17,2.390,True
8,T3 — bất lợi,cutoff=4,4,-1.00,35,2.935,True
9,T4 — bàn rỗng,full,4,0.00,7275,429.053,True


`Khớp nước tối ưu full` là tiêu chí chất lượng nước đi. Cutoff nhỏ có số nút/thời gian thấp hơn nhưng có thể khác search đầy đủ vì heuristic chỉ nhìn được cục bộ; cutoff 4 thường ổn định hơn cutoff 2.

## Heuristic agent đấu Random — 100 ván cho mỗi vai

`Tổng thời gian` cộng thời gian những lần agent thật sự chọn nước đi. `TB/nước đi` chia cho đúng `agent_moves`, không chia cho số ván.

In [6]:
tournament_rows = []
for cutoff in (2, 4):
    agent = lambda board, player, depth=cutoff: heuristic_search(board, player, cutoff=depth)
    for symbol in ("x", "o"):
        stats = play_vs_random(agent, symbol, games=100, seed=SEED + cutoff + (symbol == "o"))
        tournament_rows.append({"Agent": f"Heuristic AB cutoff={cutoff}", "Vai": symbol.upper(),
                                "Thắng": stats["win"], "Hòa": stats["draw"], "Thua": stats["loss"],
                                "Ván": stats["games"], "Nước agent": stats["agent_moves"],
                                "Tổng thời gian (s)": round(stats["total_agent_s"], 4),
                                "TB / nước agent (ms)": round(stats["mean_s_per_agent_move"] * 1_000, 4)})
tournament = pd.DataFrame(tournament_rows)
assert (tournament["Ván"] == 100).all()
tournament

,Agent,Vai,Thắng,Hòa,Thua,Ván,Nước agent,Tổng thời gian (s),TB / nước agent (ms)
0,Heuristic AB cutoff=2,X,96,4,0,100,328,1.5233,4.6443
1,Heuristic AB cutoff=2,O,89,11,0,100,322,1.1211,3.4816
2,Heuristic AB cutoff=4,X,96,4,0,100,378,11.7614,31.1149
3,Heuristic AB cutoff=4,O,83,17,0,100,351,8.7444,24.9128


### Nhận xét

* Cutoff 2 là lựa chọn rất nhanh nhưng có thể bỏ sót đe dọa ở tầng sâu hơn.
* Cutoff 4 duyệt nhiều nút hơn, đổi lại thường khớp nước đi của full search nhiều hơn.
* Khi đấu Random, vai X thường hưởng lợi từ quyền đi trước; đánh giá cần tách X/O như bảng trên.
* Trên Tic-Tac-Toe full search còn khả thi; với game lớn hơn, cutoff và heuristic là cần thiết để giữ thời gian phản hồi.